# Deep Learning Project

## Imports

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
import cv2

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torch.optim as optim
import torch.nn as nn

from models.classifiers.NN.NN_1 import DeepFakeNN_1

## Load Dataset

In [ ]:
#  Grab all the real images from the 'wiki' folder
real_images = glob("data/wiki/**/*.jpg", recursive=True)

# Grab all the fake images from the other three folders
fake_images_dict = {}
total_fake_count = 0
for folder in ["inpainting", "insight", "text2img"]:
    paths = glob(f"data/{folder}/**/*.jpg", recursive=True)
    fake_images_dict[folder] = paths
    total_fake_count += len(paths)

print(f"Total Real Images found: {len(real_images)}")
print(f"Total Fake Images found: {total_fake_count}")

for folder, paths in fake_images_dict.items():
    print(f"  - {folder}: {len(paths)}")

In [ ]:
first_5_real = real_images[:5]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
fig.suptitle("Real Images", fontsize=16)

for i, path in enumerate(first_5_real):
    img = cv2.imread(path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    axes[i].imshow(img_rgb)
    axes[i].set_title(f"Real {i+1}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
for folder_name, paths in fake_images_dict.items():

    first_5_category = paths[:5]

    fig, axes = plt.subplots(1, 5, figsize=(15, 3))
    fig.suptitle(f"Fake Images: {folder_name.capitalize()}", fontsize=16)

    for i, path in enumerate(first_5_category):
        img = cv2.imread(path)

        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[i].imshow(img_rgb)

        axes[i].set_title(f"{folder_name} {i+1}")
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

## Preprocessing

In [ ]:
class DeepFakeDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.loc[idx, 'filepath']
        label = self.dataframe.loc[idx, 'label']

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        label = torch.tensor([label], dtype=torch.float32)

        return image, label

In [ ]:
def create_preprocessing_pipeline(real_list, fake_dict, batch_size=32, img_size=(224, 224), fake_index=3):
    """
    Takes lists of real and fake images, builds dataframes, splits the data,
    and returns PyTorch DataLoaders and the split DataFrames.

    fake_index:
        0 = use first fake category
        1 = use second fake category
        2 = use third fake category
        3 = use ALL fake categories
    """

    print("Building DataFrames...")
    # Create Real DataFrame
    df_real = pd.DataFrame({'filepath': real_list, 'label': 0, 'source': 'wiki'})

    # Determine which fake folders to use based on fake_index
    folder_names = list(fake_dict.keys())

    if fake_index == 3:
        folders_to_use = folder_names
        print("Using ALL fake categories.")
    elif 0 <= fake_index <= 2:
        folders_to_use = [folder_names[fake_index]]
        print(f"Using ONLY fake category: {folders_to_use[0]}")
    else:
        raise ValueError("fake_index must be 0, 1, 2, or 3")

    # Create Fake DataFrame
    fake_dfs = []
    for folder_name in folders_to_use:
        paths = fake_dict[folder_name]
        df = pd.DataFrame({'filepath': paths, 'label': 1, 'source': folder_name})
        fake_dfs.append(df)

    df_fake = pd.concat(fake_dfs, ignore_index=True)

    # Combine into full dataset
    df_full = pd.concat([df_real, df_fake], ignore_index=True)

    print("Splitting Data (70% Train, 15% Val, 15% Test)...")
    train_df, temp_df = train_test_split(
        df_full, test_size=0.30, random_state=42, stratify=df_full['label']
    )

    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, random_state=42, stratify=temp_df['label']
    )

    print("Setting up Transforms and DataLoaders...")
    data_transform = transforms.Compose([
        transforms.Resize(img_size),
        transforms.ToTensor()
    ])

    # Create Datasets
    train_dataset = DeepFakeDataset(train_df, transform=data_transform)
    val_dataset = DeepFakeDataset(val_df, transform=data_transform)
    test_dataset = DeepFakeDataset(test_df, transform=data_transform)

    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print("Pipeline Complete!\n")

    return train_loader, val_loader, test_loader, train_df, val_df, test_df

## Model Training

In [ ]:
def train_model(model, train_loader, criterion, optimizer, device, epochs=5):
    """
    Trains a PyTorch model and returns the trained model along with its training history.
    """
    print(f"Starting training on {device} for {epochs} epochs...\n")

    history = {'loss': [], 'accuracy': []}

    for epoch in range(epochs):
        model.train()

        running_loss = 0.0
        correct_guesses = 0
        total_samples = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()               # 1. Clear old gradients
            outputs = model(images)             # 2. Forward pass
            loss = criterion(outputs, labels)   # 3. Calculate loss
            loss.backward()                     # 4. Backward pass
            optimizer.step()                    # 5. Update weights

            running_loss += loss.item()

            predictions = (torch.sigmoid(outputs) >= 0.5).float()
            correct_guesses += (predictions == labels).sum().item()
            total_samples += labels.size(0)

        # Calculate average loss and accuracy for this epoch
        epoch_loss = running_loss / len(train_loader)
        epoch_accuracy = (correct_guesses / total_samples) * 100

        # Save to history
        history['loss'].append(epoch_loss)
        history['accuracy'].append(epoch_accuracy)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_loss:.4f} | Train Accuracy: {epoch_accuracy:.2f}%")

    print("\nTraining Complete!")
    return model, history

### NN

#### NN_1

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))

all_models = {}
all_histories = {}

folder_names = list(fake_images_dict.keys())

for fake_idx in range(4):
    if fake_idx == 3:
        experiment_name = "all_fakes"
    else:
        experiment_name = folder_names[fake_idx]

    print(f"\n{'='*60}")
    print(f"🚀 STARTING EXPERIMENT: {experiment_name.upper()}")
    print(f"{'='*60}")

    # Create the data pipeline for this specific fake_idx
    train_loader, val_loader, test_loader, train_df, val_df, test_df = create_preprocessing_pipeline(
        real_list=real_images,
        fake_dict=fake_images_dict,
        batch_size=32,
        img_size=(224, 224),
        fake_index=fake_idx
    )

    print(f"Number of training batches: {len(train_loader)}")

    # Initialize the model
    model = DeepFakeNN_1().to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    trained_model, train_history = train_model(
        model=model,
        train_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        epochs=5
    )

    all_models[experiment_name] = trained_model
    all_histories[experiment_name] = train_history

print("\n🎉 ALL 4 EXPERIMENTS COMPLETED SUCESSFULLY!")